# 01 · Data cleanliness check — all workbooks, all sheets
Checks each of the 5 PS-12 workbooks sheet by sheet: shape, dtypes, nulls, duplicates, timestamp coverage, value ranges,
plus targeted tests for the **deliberate issues** listed in each README.

**Anti-leakage:** `anomaly_labels` is only checked structurally; any row-level content shown is restricted to anomalies starting **before 1 Apr 2026** (tuning window).

This notebook only *diagnoses*. It does not modify the source files.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_rows", 100)

ROOT = Path.cwd()
if not (ROOT / "PS-12_meter_data.xlsx").exists():
    ROOT = ROOT.parent
FILES = {
    "meter": "PS-12_meter_data.xlsx",
    "bms": "PS-12_bms_submeters.xlsx",
    "context": "PS-12_context_and_bills.xlsx",
    "anomaly": "PS-12_anomaly_test_set.xlsx",
}
TUNING_END = pd.Timestamp("2026-04-01")

# Load every sheet of every workbook (README sheets are kept as text, not profiled as data)
raw = {}      # (workbook, sheet) -> DataFrame
for key, fname in FILES.items():
    xl = pd.read_excel(ROOT / fname, sheet_name=None)
    for sheet, df in xl.items():
        raw[(key, sheet)] = df
print(f"{'workbook':<9}{'sheet':<30}{'rows':>8}{'cols':>6}")
for (k, s), df in raw.items():
    print(f"{k:<9}{s:<30}{len(df):>8}{df.shape[1]:>6}")

workbook sheet                             rows  cols
meter    README                              47     3
meter    gurugram_15min                   35040     8
meter    jaipur_hourly                     8760     7
meter    goa_hourly                        8784     7
bms      README                              75     3
bms      gurugram_submeters_hourly         8760    23
bms      jaipur_goa_submeters_hourly      17520    11
bms      bms_command_log                   1173     7
bms      equipment_register                  24    10
context  README                              72     3
context  weather_hourly                   26280     5
context  occupancy_daily                   1095     7
context  events                             330     8
context  tariffs                             30     5
context  monthly_bills                       36    22
anomaly  README                              45     3
anomaly  anomaly_labels                     111    13
anomaly  bms_alarm_log      

## Helpers

In [2]:
issues = []   # (workbook, sheet, severity, finding)

def flag(wb, sheet, severity, msg):
    issues.append((wb, sheet, severity, msg))
    print(f"  [{severity}] {msg}")

def data_sheets(wb):
    return {s: df for (k, s), df in raw.items() if k == wb and s != "README"}

def profile(wb, sheet, df, key_cols=None, ts_col=None):
    """Generic profile: dtypes, nulls, full-row duplicates, key duplicates, constant columns."""
    print(f"\n=== {wb} / {sheet}  ({len(df):,} rows x {df.shape[1]} cols)")
    nulls = df.isna().sum()
    out = pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": nulls,
                        "null_%": (nulls / len(df) * 100).round(2), "unique": df.nunique()})
    display(out)
    if nulls.sum():
        flag(wb, sheet, "warn", f"{int(nulls.sum())} null cells in {int((nulls > 0).sum())} column(s): {list(nulls[nulls > 0].index)}")
    n_dup = int(df.duplicated().sum())
    if n_dup:
        flag(wb, sheet, "warn", f"{n_dup} fully duplicated rows")
    if key_cols:
        n_key = int(df.duplicated(subset=key_cols).sum())
        if n_key:
            flag(wb, sheet, "warn", f"{n_key} duplicate keys on {key_cols}")
    const = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
    if const:
        flag(wb, sheet, "info", f"constant columns: {const}")
    if ts_col:
        ts = pd.to_datetime(df[ts_col], errors="coerce")
        bad = int(ts.isna().sum() - df[ts_col].isna().sum())
        if bad:
            flag(wb, sheet, "error", f"{bad} unparseable values in {ts_col}")
        print(f"  range: {ts.min()} -> {ts.max()}")

def numeric_summary(df, cols=None):
    cols = cols or df.select_dtypes("number").columns
    s = df[cols].describe().T[["min", "mean", "max"]]
    s["negatives"] = (df[cols] < 0).sum()
    s["zeros"] = (df[cols] == 0).sum()
    return s

def ts_regularity(wb, sheet, ts, freq):
    """Missing / duplicate timestamps against an expected regular grid."""
    ts = pd.to_datetime(ts).dropna()
    full = pd.date_range(ts.min(), ts.max(), freq=freq)
    dup = int(ts.duplicated().sum())
    missing = full.difference(ts.unique())
    print(f"  expected {len(full):,} @ {freq}, got {len(ts):,} rows ({ts.nunique():,} unique); missing={len(missing)}, duplicate={dup}")
    if dup:
        flag(wb, sheet, "warn", f"{dup} duplicate timestamps")
    if len(missing):
        flag(wb, sheet, "warn", f"{len(missing)} missing timestamps, first {missing[:3].tolist()}")
    return missing

def runs(mask):
    """Return (start_idx, length) of consecutive True runs in a boolean array."""
    m = np.asarray(mask, dtype=bool)
    if not m.any():
        return []
    d = np.diff(np.r_[0, m.astype(int), 0])
    return list(zip(np.where(d == 1)[0], (np.where(d == -1)[0] - np.where(d == 1)[0])))

## READMEs — the datasets' own list of known issues (for reference)

In [3]:
for (k, s), df in raw.items():
    if s == "README":
        txt = df.fillna("").astype(str).apply(lambda r: " ".join(v for v in r if v), axis=1)
        hit = txt[txt.str.startswith("•")]
        print(f"\n{FILES[k]}")
        for line in hit:
            print("  ", line)


PS-12_meter_data.xlsx
   • Gurugram has three meter communication faults (flat lines or zeros) — find and exclude them.
   • Jaipur has a 10-hour gap with blank values.
   • Goa has 24 duplicated hourly rows.
   • During DG operation the grid columns read zero; total site energy = grid_kwh + dg_kwh.

PS-12_bms_submeters.xlsx
   • In Gurugram, the lifts_kw and pool_kw sub-meters were swapped in the BMS for March 2026.
   • Sub-meters do not add up exactly to the main meter: other_kw is the unmetered remainder, and sub-meters carry ±1% error.
   • The BMS command log is thinned for chillers that switch very often; it is not a complete record of every start and stop.

PS-12_context_and_bills.xlsx
   • Goa's July 2026 bill was issued on an estimated reading and is 14% higher than the meter data supports.
   • Tariff rates are fictionalised but follow the structure of Indian HT commercial tariffs.
   • Event end times of 23:59 mean the event ran past midnight.

PS-12_anomaly_test_set.xlsx


# 1 · PS-12_meter_data.xlsx
Sheets: `gurugram_15min`, `jaipur_hourly`, `goa_hourly`.
Known issues: Gurugram comms faults (flat/zero), Jaipur 10-hour gap, Goa 24 duplicate rows, DG periods read 0 on grid columns.

In [4]:
M = data_sheets("meter")
for s, df in M.items():
    profile("meter", s, df, key_cols=["timestamp"], ts_col="timestamp")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    display(numeric_summary(df))


=== meter / gurugram_15min  (35,040 rows x 8 cols)


,dtype,nulls,null_%,unique
timestamp,datetime64[us],0,0.00,35040
grid_kwh,float64,0,0.00,10794
grid_kvah,float64,0,0.00,11131
kw_avg,float64,0,0.00,5064
kva_demand,float64,0,0.00,5215
power_factor,float64,145,0.41,85
supply_source,str,0,0.00,2
dg_kwh,float64,0,0.00,145


  [warn] 145 null cells in 1 column(s): ['power_factor']
  range: 2025-10-01 00:00:00 -> 2026-09-30 23:45:00


,min,mean,max,negatives,zeros
grid_kwh,0.00,117.826739,264.930,0,166
grid_kvah,0.00,121.510047,271.500,0,166
kw_avg,0.00,471.552840,1059.700,0,145
kva_demand,0.00,486.040297,1086.000,0,166
power_factor,0.86,0.970235,0.993,0,0
dg_kwh,0.00,0.491421,164.320,0,34895



=== meter / jaipur_hourly  (8,760 rows x 7 cols)


,dtype,nulls,null_%,unique
timestamp,datetime64[us],0,0.00,8760
grid_kwh,float64,10,0.11,1854
grid_kvah,float64,10,0.11,1906
kw_avg,float64,10,0.11,1833
kva_max_in_hour,float64,10,0.11,1971
power_factor,float64,10,0.11,71
dg_kwh,float64,0,0.00,43


... (2 outputs left out)


=== meter / goa_hourly  (8,784 rows x 7 cols)


,dtype,nulls,null_%,unique
timestamp,datetime64[us],0,0.0,8760
grid_kwh,float64,0,0.0,1909
grid_kvah,float64,0,0.0,1954
kw_avg,float64,0,0.0,1883
kva_max_in_hour,float64,0,0.0,2023
power_factor,float64,0,0.0,29
dg_kwh,float64,0,0.0,64


  [warn] 24 fully duplicated rows
  [warn] 24 duplicate keys on ['timestamp']
  range: 2025-10-01 00:00:00 -> 2026-09-30 23:00:00


,min,mean,max,negatives,zeros
grid_kwh,0.000,240.817942,387.90,0,24
grid_kvah,0.000,246.748770,399.30,0,24
kw_avg,154.900,241.987728,387.90,0,0
kva_max_in_hour,0.000,259.571778,416.10,0,24
power_factor,0.962,0.975977,0.99,0,0
dg_kwh,0.000,1.169752,350.10,0,8720


In [5]:
# Regular grid + duplicates, per sheet
ts_regularity("meter", "gurugram_15min", M["gurugram_15min"]["timestamp"], "15min")
ts_regularity("meter", "jaipur_hourly", M["jaipur_hourly"]["timestamp"], "h")
ts_regularity("meter", "goa_hourly", M["goa_hourly"]["timestamp"], "h")

  expected 35,040 @ 15min, got 35,040 rows (35,040 unique); missing=0, duplicate=0
  expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0
  expected 8,760 @ h, got 8,784 rows (8,760 unique); missing=0, duplicate=24
  [warn] 24 duplicate timestamps


DatetimeIndex([], dtype='datetime64[us]', freq='h')

In [6]:
# Gurugram: flat lines and zeros while supplied by GRID  (candidate meter comms faults)
g = M["gurugram_15min"].sort_values("timestamp").reset_index(drop=True)
grid = g["supply_source"].eq("GRID")
# Fault signature: grid_kwh, grid_kvah and kva_demand all repeat the previous value (flat line) or sit at 0 while on GRID.
# Natural load noise never repeats to 2 d.p. for >= 1 h; DG periods are excluded because grid columns are legitimately 0 there.
rep = (g[["grid_kwh", "grid_kvah", "kva_demand"]].diff() == 0).all(axis=1)
run_id = (~rep).cumsum()                                  # a flat run = first reading + its repeats
run_len = g.groupby(run_id)["grid_kwh"].transform("size")
flat = (run_len >= 4) & grid                              # >= 1 h of 15-min intervals
fault_cands = []
for start, ln in runs(flat):
    kind = "zeros on GRID" if g.loc[start, "grid_kwh"] == 0 else "flat line"
    fault_cands.append((kind, g.loc[start, "timestamp"], g.loc[start + ln - 1, "timestamp"], ln, g.loc[start, "grid_kwh"]))
fc = pd.DataFrame(fault_cands, columns=["type", "from", "to", "intervals", "stuck_grid_kwh"])
display(fc)
if len(fc):
    w0 = g[(g.timestamp >= fc.loc[0, "from"]) & (g.timestamp <= fc.loc[0, "to"])]
    print("kw_avg during first fault window (does it also freeze?):", w0.kw_avg.nunique(), "distinct values over", len(w0), "rows")
if len(fc):
    flag("meter", "gurugram_15min", "warn", f"{len(fc)} meter-fault run(s) (>=1 h) in grid_kwh/kvah/kVA ({fc['from'].dt.strftime('%d %b').tolist()}) -> mask from baseline; kw_avg and PF stay live and track the sub-meters (checked), so the billed columns are corrupted, not the load; rebuild them from kw_avg")
print("DG intervals:", int((g.supply_source == 'DG').sum()),
      "| DG rows with non-zero grid kWh:", int(((g.supply_source == 'DG') & (g.grid_kwh > 0)).sum()),
      "| GRID rows with dg_kwh>0:", int(((g.supply_source == 'GRID') & (g.dg_kwh > 0)).sum()))

,type,from,to,intervals,stuck_grid_kwh
0,flat line,2026-01-12 07:30:00,2026-01-12 19:30:00,49,121.75
1,zeros on GRID,2026-07-08 03:30:00,2026-07-08 08:30:00,21,0.00
2,flat line,2026-08-17 15:15:00,2026-08-17 23:30:00,34,170.39


kw_avg during first fault window (does it also freeze?): 49 distinct values over 49 rows
  [warn] 3 meter-fault run(s) (>=1 h) in grid_kwh/kvah/kVA (['12 Jan', '08 Jul', '17 Aug']) -> mask from baseline; kw_avg and PF stay live and track the sub-meters (checked), so the billed columns are corrupted, not the load; rebuild them from kw_avg
DG intervals: 145 | DG rows with non-zero grid kWh: 0 | GRID rows with dg_kwh>0: 0


In [7]:
# Physical consistency: kVAh >= kWh, PF in (0,1], kVA >= kW
for s, df in M.items():
    grid_rows = df[df["grid_kwh"] > 0]
    bad_kvah = int((grid_rows["grid_kvah"] < grid_rows["grid_kwh"] - 1e-6).sum())
    bad_pf = int(((grid_rows["power_factor"] > 1) | (grid_rows["power_factor"] <= 0)).sum())
    print(f"{s}: kVAh<kWh rows={bad_kvah}, PF out of (0,1]={bad_pf}, PF min={grid_rows['power_factor'].min():.3f}")
    if bad_kvah or bad_pf:
        flag("meter", s, "error", f"kVAh<kWh in {bad_kvah} rows / bad PF in {bad_pf} rows")

gurugram_15min: kVAh<kWh rows=0, PF out of (0,1]=0, PF min=0.860
jaipur_hourly: kVAh<kWh rows=0, PF out of (0,1]=0, PF min=0.865
goa_hourly: kVAh<kWh rows=0, PF out of (0,1]=0, PF min=0.962


In [8]:
# Jaipur: blank block (the 10-hour gap)
j = M["jaipur_hourly"].sort_values("timestamp").reset_index(drop=True)
blank = j["grid_kwh"].isna() | j["kw_avg"].isna()
for start, ln in runs(blank):
    print(f"Jaipur blank block: {j.loc[start,'timestamp']} for {ln} h")
    flag("meter", "jaipur_hourly", "warn", f"blank block of {ln} h starting {j.loc[start,'timestamp']} -> interpolate or exclude")

# Goa: duplicates
go = M["goa_hourly"]
dups = go[go.duplicated("timestamp", keep=False)].sort_values("timestamp")
print(f"Goa duplicate-timestamp rows: {len(dups)} ({go.duplicated('timestamp').sum()} extras); identical copies: "
      f"{go.duplicated().sum()}")
display(dups.head(6))

Jaipur blank block: 2026-02-03 09:00:00 for 10 h


  [warn] blank block of 10 h starting 2026-02-03 09:00:00 -> interpolate or exclude
Goa duplicate-timestamp rows: 48 (24 extras); identical copies: 24


,timestamp,grid_kwh,grid_kvah,kw_avg,kva_max_in_hour,power_factor,dg_kwh
428,2025-10-18 20:00:00,280.1,288.7,280.1,295.9,0.970,0.0
429,2025-10-18 20:00:00,280.1,288.7,280.1,295.9,0.970,0.0
1083,2025-11-15 02:00:00,192.4,197.3,192.4,203.1,0.976,0.0
1084,2025-11-15 02:00:00,192.4,197.3,192.4,203.1,0.976,0.0
1224,2025-11-20 22:00:00,328.7,336.8,328.7,343.5,0.976,0.0
1225,2025-11-20 22:00:00,328.7,336.8,328.7,343.5,0.976,0.0


# 2 · PS-12_bms_submeters.xlsx
Sheets: `gurugram_submeters_hourly`, `jaipur_goa_submeters_hourly`, `bms_command_log`, `equipment_register`.
Known issues: Gurugram `lifts_kw` / `pool_kw` swapped in March 2026; sub-meters ±1% and don't sum to main meter; command log is thinned.

In [9]:
B = data_sheets("bms")
profile("bms", "gurugram_submeters_hourly", B["gurugram_submeters_hourly"], key_cols=["timestamp"], ts_col="timestamp")
profile("bms", "jaipur_goa_submeters_hourly", B["jaipur_goa_submeters_hourly"], key_cols=["site", "timestamp"], ts_col="timestamp")
profile("bms", "bms_command_log", B["bms_command_log"], ts_col="timestamp")
profile("bms", "equipment_register", B["equipment_register"], key_cols=["site", "equipment_id"])


=== bms / gurugram_submeters_hourly  (8,760 rows x 23 cols)


,dtype,nulls,null_%,unique
timestamp,datetime64[us],0,0.0,8760
chiller_ch1_kw,float64,0,0.0,1474
chiller_ch2_kw,int64,0,0.0,1
chiller_ch3_kw,float64,0,0.0,445
chw_pumps_kw,float64,0,0.0,237
cw_pumps_ct_kw,float64,0,0.0,288
ahu_public_kw,float64,0,0.0,252
ahu_banquet_kw,float64,0,0.0,125
ahu_heaters_kw,float64,0,0.0,461
kitchen_kw,float64,0,0.0,601


  [info] constant columns: ['chiller_ch2_kw']


  range: 2025-10-01 00:00:00 -> 2026-09-30 23:00:00

=== bms / jaipur_goa_submeters_hourly  (17,520 rows x 11 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,2
timestamp,datetime64[us],0,0.0,8760
chillers_kw,float64,0,0.0,982
ahu_kw,float64,0,0.0,91
kitchen_kw,float64,0,0.0,575
laundry_kw,float64,0,0.0,251
lighting_kw,float64,0,0.0,148
pumps_pool_kw,float64,0,0.0,41
rooms_kw,float64,0,0.0,267
other_kw,float64,0,0.0,182


  range: 2025-10-01 00:00:00 -> 2026-09-30 23:00:00



=== bms / bms_command_log  (1,173 rows x 7 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,1
timestamp,datetime64[us],0,0.0,692
point,str,0,0.0,4
old_value,float64,0,0.0,4
new_value,float64,0,0.0,5
changed_by,str,0,0.0,5
source,str,0,0.0,3


  [info] constant columns: ['site']
  range: 2025-10-01 09:00:00 -> 2026-09-30 06:00:00

=== bms / equipment_register  (24 rows x 10 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,3
equipment_id,str,0,0.0,20
description,str,0,0.0,21
rating,str,0,0.0,23
connected_kw,int64,0,0.0,18
efficiency_note,str,0,0.0,10
year_installed,int64,0,0.0,7
serves,str,0,0.0,19
bms_controllable,str,0,0.0,8
operating_limit,str,0,0.0,18


In [10]:
gs = B["gurugram_submeters_hourly"].copy()
gs["timestamp"] = pd.to_datetime(gs["timestamp"])
display(numeric_summary(gs))
ts_regularity("bms", "gurugram_submeters_hourly", gs["timestamp"], "h")
jg = B["jaipur_goa_submeters_hourly"].copy()
jg["timestamp"] = pd.to_datetime(jg["timestamp"])
for site, d in jg.groupby("site"):
    print(site, end=" ")
    ts_regularity("bms", f"jaipur_goa_submeters_hourly[{site}]", d["timestamp"], "h")
display(numeric_summary(jg))

,min,mean,max,negatives,zeros
chiller_ch1_kw,0.0,71.164874,290.6,0,4089
chiller_ch2_kw,0.0,0.000000,0.0,0,8760
chiller_ch3_kw,0.0,29.351050,91.6,0,4114
chw_pumps_kw,14.1,21.142580,39.5,0,0
cw_pumps_ct_kw,11.7,19.789361,46.0,0,0
ahu_public_kw,24.9,51.499007,67.3,0,0
ahu_banquet_kw,0.0,10.741142,137.0,0,6844
ahu_heaters_kw,0.0,3.905205,273.6,0,7455
kitchen_kw,17.4,44.636861,116.4,0,0
kitchen_exhaust_mua_kw,0.0,16.964098,44.7,0,2016


  expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0


GOA   expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0
JAI   expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0


,min,mean,max,negatives,zeros
chillers_kw,33.2,66.828590,156.1,0,0
ahu_kw,19.3,32.354960,55.4,0,0
kitchen_kw,13.5,32.281667,98.0,0,0
laundry_kw,0.0,14.863094,63.1,0,12512
lighting_kw,15.4,28.238647,91.6,0,0
pumps_pool_kw,9.6,15.519406,24.8,0,0
rooms_kw,10.5,23.972386,38.6,0,0
other_kw,22.6,31.977888,42.7,0,0
chiller_load_tr,39.0,87.170548,182.0,0,0


In [11]:
# Lifts vs pool swap: monthly mean of each. Lifts ~ small & flat-ish; pool has own level. Look for March reversal.
mm = gs.set_index("timestamp")[["lifts_kw", "pool_kw"]].resample("MS").mean().round(2)
mm["lifts_vs_median_%"] = ((mm.lifts_kw / mm.lifts_kw.median() - 1) * 100).round(0)
display(mm)
# Pool is genuinely seasonal (winter heating), so test the stable meter: lifts should hover near its median every month.
odd = mm[mm["lifts_vs_median_%"].abs() > 25]
if len(odd):
    flag("bms", "gurugram_submeters_hourly", "warn", f"lifts_kw level abnormal in {[d.strftime('%Y-%m') for d in odd.index]} (pool_kw carries the lifts level) -> lifts/pool swapped; swap back")

,lifts_kw,pool_kw,lifts_vs_median_%
timestamp,,,
2025-10-01,14.90,8.00,1.0
2025-11-01,15.12,8.00,3.0
2025-12-01,14.51,26.00,-1.0
2026-01-01,14.85,25.98,1.0
2026-02-01,15.24,26.02,4.0
2026-03-01,8.00,15.02,-46.0
2026-04-01,14.91,8.00,1.0
2026-05-01,14.47,8.00,-2.0
2026-06-01,14.32,8.00,-3.0


  [warn] lifts_kw level abnormal in ['2026-03'] (pool_kw carries the lifts level) -> lifts/pool swapped; swap back


In [12]:
# Chiller consistency: chillers_running vs number of chillers with kW > 0; kW/TR efficiency sanity
ch = gs[["chiller_ch1_kw", "chiller_ch2_kw", "chiller_ch3_kw"]]
n_on = (ch > 1).sum(axis=1)
mismatch = (n_on != gs["chillers_running"])
print("chillers_running != count(kW>1):", int(mismatch.sum()), "hours")
if mismatch.sum():
    flag("bms", "gurugram_submeters_hourly", "info", f"{int(mismatch.sum())} h where chillers_running disagrees with chiller kW")
loaded = gs[gs["chiller_load_tr"] > 20].copy()
loaded["kw_per_tr"] = ch.loc[loaded.index].sum(axis=1) / loaded["chiller_load_tr"]
print("kW/TR (load>20 TR) quantiles:"); display(loaded["kw_per_tr"].describe(percentiles=[.01, .5, .99]).round(3))
print("CHW setpoint values:", sorted(gs["chw_setpoint_c"].dropna().unique())[:12])
low_sp = gs.loc[gs.chw_setpoint_c < 6.5, "timestamp"]
print("setpoint < 6.5 C (operational breach of guide limit, not a data error):", len(low_sp), "h",
      f"| first {low_sp.min()} last {low_sp.max()}")
if len(low_sp):
    flag("bms", "gurugram_submeters_hourly", "info", f"{len(low_sp)} h with CHW setpoint 5.5 C (< 6.5 C guide limit) - real operational waste, keep in data; do not 'clean'")

chillers_running != count(kW>1): 0 hours
kW/TR (load>20 TR) quantiles:


count    8760.000
mean        0.742
std         0.213
min         0.575
1%          0.585
50%         0.667
99%         1.618
max         1.861
Name: kw_per_tr, dtype: float64

CHW setpoint values: [np.float64(5.5), np.float64(7.0)]
setpoint < 6.5 C (operational breach of guide limit, not a data error): 1912 h | first 2026-05-13 08:00:00 last 2026-09-30 23:00:00
  [info] 1912 h with CHW setpoint 5.5 C (< 6.5 C guide limit) - real operational waste, keep in data; do not 'clean'


In [13]:
# Sub-meter total vs main meter (hourly), Gurugram
gm = M["gurugram_15min"].copy()
main_h = gm.set_index("timestamp")["kw_avg"].resample("h").mean()
sub_cols = [c for c in gs.columns if c.endswith("_kw")]
sub_sum = gs.set_index("timestamp")[sub_cols].sum(axis=1)
cmp_ = pd.concat([main_h.rename("main"), sub_sum.rename("subs")], axis=1).dropna()
cmp_ = cmp_[cmp_.main > 0]
rel = (cmp_.subs - cmp_.main) / cmp_.main
print(f"sub-sum vs main: median diff {rel.median():.2%}, 5-95% [{rel.quantile(.05):.2%}, {rel.quantile(.95):.2%}]  (incl. other_kw)")
big = (rel.abs() > 0.1).sum()
print("hours where |diff| > 10%:", int(big), "(these overlap meter-fault periods if main meter was flat/zero)")

sub-sum vs main: median diff 0.01%, 5-95% [-2.84%, 3.09%]  (incl. other_kw)


hours where |diff| > 10%: 28 (these overlap meter-fault periods if main meter was flat/zero)


In [14]:
# BMS command log
cl = B["bms_command_log"].copy()
cl["timestamp"] = pd.to_datetime(cl["timestamp"])
print(cl.groupby(["site", "point"]).size().unstack(0).fillna(0).astype(int))
print("changed_by:", cl.changed_by.value_counts().to_dict())
print("source:", cl.source.value_counts().to_dict())
print("hand-mode (new_value==2) entries:", int((cl.new_value == 2).sum()))
unsorted = not cl["timestamp"].is_monotonic_increasing
print("sorted by time:", not unsorted)
# old_value should equal the previous new_value per point (continuity); thinned log will break this
cl = cl.sort_values(["site", "point", "timestamp"])
cont = cl.groupby(["site", "point"]).apply(lambda d: (d["old_value"].values[1:] != d["new_value"].values[:-1]).sum(), include_groups=False)
print("continuity breaks old_value != previous new_value:\n", cont[cont > 0])
display(B["equipment_register"])

site                     GGN
point                       
Banquet AHU mode          16
CHILLER_CH1 run command  481
CHILLER_CH3 run command  665
CHW supply setpoint       11
changed_by: {'BMS sequence': 985, 'Manual (hand mode)': 161, 'Banquet supervisor': 16, 'Night shift engineer': 6, 'Duty engineer': 5}
source: {'BMS': 1146, 'Local panel (hand mode)': 16, 'BMS workstation': 11}
hand-mode (new_value==2) entries: 16
sorted by time: True
continuity breaks old_value != previous new_value:
 site  point           
GGN   Banquet AHU mode    15
dtype: int64


,site,equipment_id,description,rating,connected_kw,efficiency_note,year_installed,serves,bms_controllable,operating_limit
0,GGN,CH-1,Water-cooled screw chiller,300 TR,186,0.62 kW/TR at full load,2017,"Guest rooms, public areas",Yes,Never below 6.5 °C leaving water
1,GGN,CH-2,Water-cooled screw chiller,300 TR,186,0.62 kW/TR at full load,2017,"Guest rooms, public areas",Yes,Never below 6.5 °C leaving water
2,GGN,CH-3,Water-cooled scroll chiller (night/low load),150 TR,98,0.65 kW/TR,2021,Base load,Yes,Lead chiller below 130 TR
3,GGN,CHWP-1..3,Chilled-water secondary pumps (VFD),3 × 22 kW,66,VFD,2017,CHW loop,Yes,Min. differential pressure 1.2 bar
4,GGN,CWP-1..3 / CT-1..2,Condenser pumps and cooling towers,3 × 30 kW + 2 × 15 kW,120,—,2017,Condenser loop,Yes,Legionella dosing interlock
5,GGN,AHU-L1..L6,Public-area AHUs,6 × 7.5–15 kW,62,VFD,2017,"Lobby, restaurants, spa",Yes,Min. fresh air per NBC
6,GGN,AHU-BQ-A/B,Banquet AHUs,2 × 45 kW,90,VFD,2019,Banquet Halls A and B,Yes,Run 1 h before events
7,GGN,HTR-AHU,AHU electric re-heat banks,6 × 15 kW,90,—,2017,"Public areas, winter",Yes,Only below 16 °C outdoor
8,GGN,KEF/MUA,Kitchen exhaust and make-up air,22 kW,22,—,2017,Main kitchen,Partly,Must run whenever cooking — never automate off...
9,GGN,LAUNDRY,Washer-extractors and tumble dryers,≈ 95 kW connected,95,—,2018,In-house laundry,No (manual),Flexible load; can be scheduled


In [15]:
# Follow-up 1: are the big sub-meter vs main-meter gaps explained? (|diff| > 10%)
big = cmp_[rel.abs() > 0.10]
dg_h = (M["gurugram_15min"].set_index("timestamp").supply_source.eq("DG").astype(int).resample("h").max() > 0)
in_dg = dg_h.reindex(big.index).fillna(False)
in_fault = pd.Series(False, index=big.index)
for _, r in fc.iterrows():
    in_fault |= (big.index >= r["from"].floor("h")) & (big.index <= r["to"])
print(f"{len(big)} large-gap hours | on diesel (main kw_avg reads 0): {int(in_dg.sum())} | inside meter-fault windows: {int(in_fault.sum())} | unexplained: {int((~in_dg & ~in_fault).sum())}")
if (~in_dg & ~in_fault).sum():
    flag("bms", "gurugram_submeters_hourly", "warn", f"{int((~in_dg & ~in_fault).sum())} sub-vs-main gaps >10% unexplained by DG or meter faults")
else:
    flag("bms", "gurugram_submeters_hourly", "info", "all >10% sub-vs-main gaps fall in diesel hours (main kw_avg=0 on DG); exclude DG hours when comparing")

28 large-gap hours | on diesel (main kw_avg reads 0): 28 | inside meter-fault windows: 0 | unexplained: 0
  [info] all >10% sub-vs-main gaps fall in diesel hours (main kw_avg=0 on DG); exclude DG hours when comparing


In [16]:
# Follow-up 2: text-field consistency (whitespace, case variants, small category sets) across all data sheets
txt_issues = []
for (wb, sheet), df in raw.items():
    if sheet == "README":
        continue
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            s = df[c].dropna().astype(str)
            if (s != s.str.strip()).any() or s.str.contains(r"\s{2,}").any():
                txt_issues.append((sheet, c, "stray whitespace"))
            u = s.unique()
            if len(u) < 60 and len({x.lower().strip() for x in u}) < len(u):
                txt_issues.append((sheet, c, "case/spacing variants of same value"))
print("text issues:", txt_issues or "none")
if txt_issues:
    flag("all", "multiple", "warn", f"text inconsistencies: {txt_issues}")
# BMS command log: hand-mode entries never have a matching return-to-auto entry
cl2 = B["bms_command_log"]
print("Banquet AHU mode transitions seen:", cl2[cl2.point == "Banquet AHU mode"][["old_value", "new_value"]].drop_duplicates().values.tolist())
flag("bms", "bms_command_log", "info", "Banquet AHU mode log only records 1->2 (to hand mode); returns to auto are not logged - infer hand-mode end from sub-meter kW/schedule")

text issues:

 none


Banquet AHU mode transitions seen: [[1.0, 2.0]]
  [info] Banquet AHU mode log only records 1->2 (to hand mode); returns to auto are not logged - infer hand-mode end from sub-meter kW/schedule


# 3 · PS-12_context_and_bills.xlsx
Sheets: `weather_hourly`, `occupancy_daily`, `events`, `tariffs`, `monthly_bills`.
Known issues: Goa July 2026 bill provisional (+14%); event end 23:59 = past midnight.

In [17]:
C = data_sheets("context")
profile("context", "weather_hourly", C["weather_hourly"], key_cols=["site", "timestamp"], ts_col="timestamp")
profile("context", "occupancy_daily", C["occupancy_daily"], key_cols=["site", "date"], ts_col="date")
profile("context", "events", C["events"])
profile("context", "tariffs", C["tariffs"])
profile("context", "monthly_bills", C["monthly_bills"], key_cols=["site", "billing_month"])


=== context / weather_hourly  (26,280 rows x 5 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,3
timestamp,datetime64[us],0,0.0,8760
temp_c,float64,0,0.0,406
rel_humidity_pct,int64,0,0.0,89
rain_mm,float64,0,0.0,156


  range: 2025-10-01 00:00:00 -> 2026-09-30 23:00:00

=== context / occupancy_daily  (1,095 rows x 7 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,3
date,datetime64[us],0,0.0,365
rooms_available,int64,0,0.0,3
rooms_sold,int64,0,0.0,162
occupancy_pct,float64,0,0.0,258
guests_in_house,int64,0,0.0,263
covers_served_fnb,int64,0,0.0,354


  range: 2025-10-01 00:00:00 -> 2026-09-30 00:00:00

=== context / events  (330 rows x 8 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,3
date,datetime64[us],0,0.0,235
event_type,str,0,0.0,5
venue,str,0,0.0,6
guests,int64,0,0.0,246
start_time,str,0,0.0,3
end_time,str,0,0.0,3
catering,str,0,0.0,2



=== context / tariffs  (30 rows x 5 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,3
component,str,0,0.0,10
unit,str,0,0.0,5
rate,float64,0,0.0,23
applies,str,0,0.0,11



=== context / monthly_bills  (36 rows x 22 cols)


,dtype,nulls,null_%,unique
site,str,0,0.0,3
billing_month,str,0,0.0,12
grid_kwh,int64,0,0.0,36
grid_kvah,int64,0,0.0,36
avg_power_factor,float64,0,0.0,5
recorded_md_kva,int64,0,0.0,35
contract_demand_kva,int64,0,0.0,3
billing_demand_kva,int64,0,0.0,11
peak_window_kvah,int64,0,0.0,36
offpeak_window_kvah,int64,0,0.0,36


In [18]:
w = C["weather_hourly"].copy(); w["timestamp"] = pd.to_datetime(w["timestamp"])
for site, d in w.groupby("site"):
    print(site, end=" "); ts_regularity("context", f"weather_hourly[{site}]", d["timestamp"], "h")
display(w.groupby("site")[["temp_c", "rel_humidity_pct", "rain_mm"]].agg(["min", "mean", "max"]).round(1))
out_rng = int(((w.rel_humidity_pct < 0) | (w.rel_humidity_pct > 100) | (w.temp_c < -5) | (w.temp_c > 50) | (w.rain_mm < 0)).sum())
print("physically implausible weather rows:", out_rng)
if out_rng: flag("context", "weather_hourly", "error", f"{out_rng} implausible weather rows")

GGN   expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0
GOA   expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0
JAI   expected 8,760 @ h, got 8,760 rows (8,760 unique); missing=0, duplicate=0


temp_c             rel_humidity_pct            rain_mm           
        min  mean   max              min  mean  max     min mean   max
site                                                                  
GGN     3.1  25.2  44.4               12  56.7  100     0.0  0.1  14.2
GOA    15.5  27.4  37.7               33  76.0  100     0.0  0.4  27.1
JAI     2.9  25.1  44.9               12  51.1  100     0.0  0.1  15.0

physically implausible weather rows: 0


In [19]:
o = C["occupancy_daily"].copy(); o["date"] = pd.to_datetime(o["date"])
for site, d in o.groupby("site"):
    full = pd.date_range(d.date.min(), d.date.max(), freq="D")
    print(site, "days", len(d), "missing", len(full.difference(d.date)))
over = int((o.rooms_sold > o.rooms_available).sum())
pct_err = int(((o.rooms_sold / o.rooms_available - o.occupancy_pct).abs() > 0.005).sum())
print("rooms_sold > available:", over, "| occupancy_pct inconsistent:", pct_err)
display(o.groupby("site")[["rooms_available", "occupancy_pct", "guests_in_house", "covers_served_fnb"]].agg(["min", "mean", "max"]).round(2))
if over or pct_err: flag("context", "occupancy_daily", "error", f"rooms_sold>available {over}; pct inconsistent {pct_err}")

GGN days 365 missing 0
GOA days 365 missing 0
JAI days 365 missing 0
rooms_sold > available: 0 | occupancy_pct inconsistent: 0


rooms_available             occupancy_pct             guests_in_house              covers_served_fnb             
                 min   mean  max           min  mean   max             min    mean  max               min    mean  max
site                                                                                                                  
GGN              220  220.0  220          0.40  0.70  0.95             125  254.97  378               222  407.85  605
GOA              140  140.0  140          0.25  0.67  0.99              60  155.32  258               121  287.36  468
JAI              160  160.0  160          0.31  0.69  0.99              86  181.99  297               154  317.68  524

In [20]:
ev = C["events"].copy(); ev["date"] = pd.to_datetime(ev["date"])
print(ev.groupby(["site", "event_type"]).size().unstack(0).fillna(0).astype(int))
et = ev["end_time"].astype(str)
st = ev["start_time"].astype(str)
past_mid = et.str.startswith("23:59")
print("events with end 23:59 (ran past midnight):", int(past_mid.sum()))
if past_mid.any(): flag("context", "events", "info", f"{int(past_mid.sum())} events end '23:59' = actually past midnight; true end unknown")
neg = ((et < st) & ~past_mid).sum()
print("end before start (excluding 23:59):", int(neg), "| guests <= 0:", int((ev.guests <= 0).sum()), "| catering:", ev.catering.value_counts().to_dict())
print("events on a day with no occupancy row:", int((~ev.set_index(['site','date']).index.isin(o.set_index(['site','date']).index)).sum()))
print("multiple events same site/venue/day:", int(ev.duplicated(['site','date','venue']).sum()))

site                    GGN  GOA  JAI
event_type                           
Beach wedding / MICE      0   51    0
Corporate conference    123    0    0
Corporate offsite         0    0   40
Wedding (lawn)            0    0   72
Wedding / social event   44    0    0
events with end 23:59 (ran past midnight): 167
  [info] 167 events end '23:59' = actually past midnight; true end unknown
end before start (excluding 23:59): 0 | guests <= 0: 0 | catering: {'Yes': 310, 'No': 20}
events on a day with no occupancy row: 0
multiple events same site/venue/day: 0


In [21]:
display(C["tariffs"])
# Demand / bills: recompute totals and check against reported
b = C["monthly_bills"].copy()
comp = ["energy_charge_inr", "tod_net_inr", "demand_charge_inr", "excess_demand_penalty_inr", "fsa_inr", "electricity_duty_inr", "municipal_tax_inr"]
b["recomputed_total"] = b[comp].sum(axis=1)
b["total_diff"] = (b["recomputed_total"] - b["total_grid_bill_inr"]).round(0)
b["total_energy_diff"] = (b["total_grid_bill_inr"] + b["dg_cost_inr"] - b["total_energy_cost_inr"]).round(0)
b["pf_check"] = (b.grid_kwh / b.grid_kvah - b.avg_power_factor).abs().round(4)
b["md_over_cd"] = b.recorded_md_kva - b.contract_demand_kva
print("max |recomputed - reported| total:", b.total_diff.abs().max(), "| max PF mismatch:", b.pf_check.max())
display(b[["site", "billing_month", "bill_status", "total_grid_bill_inr", "total_diff", "total_energy_diff", "pf_check", "md_over_cd", "excess_demand_penalty_inr"]])
bad_tot = b[(b.total_diff.abs() > 5) | (b.pf_check > 0.005)]
print("rows where components != total (>Rs5) or kWh/kVAh != reported PF:", bad_tot[["site", "billing_month", "bill_status", "total_diff", "pf_check"]].values.tolist())
if len(bad_tot): flag("context", "monthly_bills", "warn", f"internally inconsistent bill rows (components/total/PF): {bad_tot[['site','billing_month']].values.tolist()} - status {bad_tot.bill_status.unique().tolist()}")
print(b.bill_status.value_counts().to_dict())
prov = b[b.bill_status != "Actual"]
display(prov[["site", "billing_month", "bill_status"]])
if len(prov): flag("context", "monthly_bills", "warn", f"{len(prov)} non-Actual bill(s): {prov[['site','billing_month']].values.tolist()}")

,site,component,unit,rate,applies
0,GGN,Energy charge,₹ per kVAh,7.100,All grid kVAh
1,GGN,Peak ToD surcharge,₹ per kVAh,1.000,18:00–22:00 every day
2,GGN,Off-peak ToD rebate,₹ per kVAh,-0.750,23:00–05:00 every day
3,GGN,Demand charge,₹ per kVA per month,290.000,"Billing demand = max(recorded MD, 75% of contr..."
4,GGN,Excess demand penalty,₹ per kVA per month,580.000,On recorded MD above contract demand
5,GGN,Fuel surcharge adjustment (FSA),₹ per kVAh,0.470,All grid kVAh (varies by quarter; average shown)
6,GGN,Electricity duty,₹ per kVAh,0.100,All grid kVAh
7,GGN,Municipal tax,% of energy + demand charges,0.020,Monthly
8,GGN,Contract demand,kVA,1000.000,Sanctioned
9,GGN,DG generation cost,₹ per kWh,27.560,"Diesel ₹92/litre at 3.6 kWh/litre (Jaipur, Goa..."


max |recomputed - reported| total: 18665 | max PF mismatch: 0.1199


,site,billing_month,bill_status,total_grid_bill_inr,total_diff,total_energy_diff,pf_check,md_over_cd,excess_demand_penalty_inr
0,GGN,2025-10,Actual,3033507,0,1,0.0002,-210,0
1,GGN,2025-11,Actual,2652478,0,0,0.0001,-266,0
2,GGN,2025-12,Actual,2761827,0,0,0.0003,-285,0
3,GGN,2026-01,Actual,2948244,0,0,0.0000,-148,0
4,GGN,2026-02,Actual,2598399,1,0,0.0000,-302,0
5,GGN,2026-03,Actual,2804710,0,0,0.0003,-295,0
6,GGN,2026-04,Actual,3039899,-1,1,0.0000,-122,0
7,GGN,2026-05,Actual,3524186,0,0,0.0005,86,49880
8,GGN,2026-06,Actual,3427393,-1,1,0.0001,66,38338
9,GGN,2026-07,Actual,3346396,0,0,0.0005,-133,0


rows where components != total (>Rs5) or kWh/kVAh != reported PF: [['GOA', '2026-07', 'Provisional (estimated reading)', -18665, 0.1199]]
  [warn] internally inconsistent bill rows (components/total/PF): [['GOA', '2026-07']] - status ['Provisional (estimated reading)']
{'Actual': 35, 'Provisional (estimated reading)': 1}


,site,billing_month,bill_status
33,GOA,2026-07,Provisional (estimated reading)


  [warn] 1 non-Actual bill(s): [['GOA', '2026-07']]


In [22]:
# Reconcile bills with meter data: monthly grid kWh / kVAh / max kVA from meters vs bill
def monthly_from_meter(df, kva_col, site):
    d = df.copy(); d["month"] = d.timestamp.dt.to_period("M").astype(str)
    return d.groupby("month").agg(meter_kwh=("grid_kwh", "sum"), meter_kvah=("grid_kvah", "sum"), meter_md=(kva_col, "max")).assign(site=site)

mg = M["gurugram_15min"]; mj = M["jaipur_hourly"].drop_duplicates("timestamp"); mgo = M["goa_hourly"].drop_duplicates("timestamp")
meter_m = pd.concat([monthly_from_meter(mg, "kva_demand", "GGN"), monthly_from_meter(mj, "kva_max_in_hour", "JAI"),
                     monthly_from_meter(mgo, "kva_max_in_hour", "GOA")]).reset_index().rename(columns={"index": "month"})
rec = b.merge(meter_m, left_on=["site", "billing_month"], right_on=["site", "month"], how="left")
rec["kwh_diff_%"] = ((rec.grid_kwh - rec.meter_kwh) / rec.meter_kwh * 100).round(2)
rec["kvah_diff_%"] = ((rec.grid_kvah - rec.meter_kvah) / rec.meter_kvah * 100).round(2)
rec["md_diff"] = (rec.recorded_md_kva - rec.meter_md).round(1)
display(rec[["site", "billing_month", "bill_status", "kwh_diff_%", "kvah_diff_%", "md_diff"]])
worst = rec.loc[rec["kvah_diff_%"].abs().nlargest(3).index, ["site", "billing_month", "bill_status", "kvah_diff_%"]]
print("largest bill-vs-meter kVAh gaps:"); display(worst)
print("(Gurugram months containing meter faults may show a gap due to flat/zero data; Goa July 2026 should stand out ~ +14%)")

,site,billing_month,bill_status,kwh_diff_%,kvah_diff_%,md_diff
0,GGN,2025-10,Actual,0.00,-0.00,-0.3
1,GGN,2025-11,Actual,-0.00,-0.00,-0.2
2,GGN,2025-12,Actual,-0.00,-0.00,0.1
3,GGN,2026-01,Actual,-0.00,0.00,0.2
4,GGN,2026-02,Actual,0.00,-0.00,0.0
5,GGN,2026-03,Actual,-0.00,0.00,0.3
6,GGN,2026-04,Actual,-0.00,-0.00,-0.1
7,GGN,2026-05,Actual,-0.00,-0.00,0.0
8,GGN,2026-06,Actual,-0.00,-0.00,-0.1
9,GGN,2026-07,Actual,-0.00,-0.00,-0.2


largest bill-vs-meter kVAh gaps:


,site,billing_month,bill_status,kvah_diff_%
33,GOA,2026-07,Provisional (estimated reading),14.00
16,JAI,2026-02,Actual,1.56
0,GGN,2025-10,Actual,-0.00


(Gurugram months containing meter faults may show a gap due to flat/zero data; Goa July 2026 should stand out ~ +14%)


# 4 · PS-12_anomaly_test_set.xlsx
Sheets: `anomaly_labels`, `bms_alarm_log`.
**Leakage guard:** only schema/nulls/ID checks over all rows; any content shown is limited to anomalies starting before 1 Apr 2026.

In [23]:
A = data_sheets("anomaly")
lab = A["anomaly_labels"].copy()
for c in ["start", "end"]:
    lab[c] = pd.to_datetime(lab[c], errors="coerce")
print(f"anomaly_labels: {len(lab)} rows | null cells per column (structural check only):")
display(lab.isna().sum()[lambda s: s > 0].to_frame("nulls"))
print("ids unique:", lab.anomaly_id.is_unique, "| end<start:", int((lab.end < lab.start).sum()),
      "| sites:", sorted(lab.site.dropna().unique()))
print("rows by window -> tuning (<1 Apr 2026):", int((lab.start < TUNING_END).sum()), "| scoring (>=1 Apr 2026):", int((lab.start >= TUNING_END).sum()))
print("severity counts (all):", lab.severity.value_counts().to_dict())
tune = lab[lab.start < TUNING_END]
print("\nTuning-window anomaly types:"); display(tune.anomaly_type.value_counts().to_frame("n"))
display(tune.head(5)[["anomaly_id", "site", "anomaly_type", "start", "end", "equipment_or_meter", "severity"]])

anomaly_labels: 111 rows | null cells per column (structural check only):


,nulls


ids unique: True | end<start: 0 | sites: ['GGN', 'GOA', 'JAI']
rows by window -> tuning (<1 Apr 2026): 40 | scoring (>=1 Apr 2026): 71
severity counts (all): {'Medium': 45, 'Low': 34, 'High': 28, 'Data quality': 4}

Tuning-window anomaly types:


,n
anomaly_type,
EVENT_LIGHTING_LEFT_ON,9
SIMULTANEOUS_HEAT_COOL,9
LAUNDRY_PEAK_TARIFF,7
AHU_SCHEDULE_OVERRIDE,6
KITCHEN_EXHAUST_LEFT_ON,3
METER_DATA_FAULT,2
CHILLER_ON_LOW_LOAD,2
POOL_PUMPS_24H,1
POWER_FACTOR_DROP,1


,anomaly_id,site,anomaly_type,start,end,equipment_or_meter,severity
0,AN-1001,GGN,AHU_SCHEDULE_OVERRIDE,2025-10-01 18:00:00,2025-10-03 09:00:00,ahu_banquet_kw,Medium
1,AN-1002,GOA,POOL_PUMPS_24H,2025-10-06 00:00:00,2025-10-28 00:00:00,pumps_pool_kw,Low
2,AN-1003,GGN,AHU_SCHEDULE_OVERRIDE,2025-10-06 18:00:00,2025-10-07 13:00:00,ahu_banquet_kw,Medium
3,AN-1004,JAI,LAUNDRY_PEAK_TARIFF,2025-10-08 18:00:00,2025-10-08 22:00:00,laundry_kw,Low
4,AN-1005,GGN,KITCHEN_EXHAUST_LEFT_ON,2025-10-10 00:00:00,2025-10-13 06:00:00,kitchen_exhaust_mua_kw,Low


In [24]:
al = A["bms_alarm_log"].copy()
profile("anomaly", "bms_alarm_log", al, key_cols=["alarm_id"])
for c in ["raised_at", "acknowledged_at", "action_at"]:
    al[c] = pd.to_datetime(al[c], errors="coerce")
print("ack before raised:", int((al.acknowledged_at < al.raised_at).sum()), "| action before raised:", int((al.action_at < al.raised_at).sum()))
print("action_taken:", al.action_taken.value_counts().to_dict())
print("action_taken='no action' but action_at filled:", int((al.action_taken.str.contains('no action', case=False, na=False) & al.action_at.notna()).sum()))
display(al.alarm.value_counts().to_frame("n"))
print("date range:", al.raised_at.min(), "->", al.raised_at.max(), "| sites:", al.site.value_counts().to_dict())


=== anomaly / bms_alarm_log  (1,829 rows x 9 cols)


,dtype,nulls,null_%,unique
alarm_id,str,0,0.00,1829
site,str,0,0.00,3
raised_at,datetime64[us],0,0.00,1807
alarm,str,0,0.00,9
value,float64,0,0.00,1459
threshold,str,0,0.00,6
acknowledged_at,datetime64[us],0,0.00,1829
action_taken,str,0,0.00,7
action_at,datetime64[us],1202,65.72,627


  [warn] 1202 null cells in 1 column(s): ['action_at']
ack before raised: 0 | action before raised: 0
action_taken: {'Acknowledged — no action': 720, 'Checked, normal for event': 254, 'Checked, normal for weather': 228, 'Raised maintenance ticket': 227, 'Equipment switched off': 188, 'Setpoint reset': 107, 'Escalated to chief engineer': 105}
action_taken='no action' but action_at filled: 0


,n
alarm,
Sensor out of range,363
BMS controller comms fail,339
AHU filter dirty,332
Supply air temperature high,316
CHW supply temperature low,154
Power factor low,128
Chiller kW high,99
Night base load high,70
Main incomer kW high,28


date range: 2025-10-01 07:19:00 -> 2026-09-30 18:04:00 | sites: {'GGN': 901, 'JAI': 487, 'GOA': 441}


# 5 · Cross-file checks

In [25]:
# Common time grid & site coverage across files
spans = {
    "meter GGN": (M["gurugram_15min"].timestamp.min(), M["gurugram_15min"].timestamp.max()),
    "meter JAI": (M["jaipur_hourly"].timestamp.min(), M["jaipur_hourly"].timestamp.max()),
    "meter GOA": (M["goa_hourly"].timestamp.min(), M["goa_hourly"].timestamp.max()),
    "sub GGN": (gs.timestamp.min(), gs.timestamp.max()),
    "sub JAI/GOA": (jg.timestamp.min(), jg.timestamp.max()),
    "weather": (w.timestamp.min(), w.timestamp.max()),
    "occupancy": (o.date.min(), o.date.max()),
    "events": (ev.date.min(), ev.date.max()),
}
display(pd.DataFrame(spans, index=["start", "end"]).T)
print("sites in weather/occupancy/events/bills:", {n: sorted(x.site.unique()) for n, x in
      [("weather", w), ("occupancy", o), ("events", ev), ("bills", b)]})
# Event guests vs capacity sanity: events on days with zero rooms sold would be odd, check rooms_sold 0 days
print("days with 0 rooms sold:", int((o.rooms_sold == 0).sum()))

,start,end
meter GGN,2025-10-01,2026-09-30 23:45:00
meter JAI,2025-10-01,2026-09-30 23:00:00
meter GOA,2025-10-01,2026-09-30 23:00:00
sub GGN,2025-10-01,2026-09-30 23:00:00
sub JAI/GOA,2025-10-01,2026-09-30 23:00:00
weather,2025-10-01,2026-09-30 23:00:00
occupancy,2025-10-01,2026-09-30 00:00:00
events,2025-10-01,2026-09-29 00:00:00


sites in weather/occupancy/events/bills: {'weather': ['GGN', 'GOA', 'JAI'], 'occupancy': ['GGN', 'GOA', 'JAI'], 'events': ['GGN', 'GOA', 'JAI'], 'bills': ['GGN', 'GOA', 'JAI']}
days with 0 rooms sold: 0


# 6 · Summary of findings

In [26]:
summ = pd.DataFrame(issues, columns=["workbook", "sheet", "severity", "finding"])
order = {"error": 0, "warn": 1, "info": 2}
summ = summ.sort_values(["severity", "workbook", "sheet"], key=lambda s: s.map(order) if s.name == "severity" else s)
display(summ.style.set_properties(subset=["finding"], **{"text-align": "left", "white-space": "pre-wrap"}))
print(summ.severity.value_counts().to_dict())

,workbook,sheet,severity,finding
16,anomaly,bms_alarm_log,warn,1202 null cells in 1 column(s): ['action_at']
9,bms,gurugram_submeters_hourly,warn,lifts_kw level abnormal in ['2026-03'] (pool_kw carries the lifts level) -> lifts/pool swapped; swap back
14,context,monthly_bills,warn,"internally inconsistent bill rows (components/total/PF): [['GOA', '2026-07']] - status ['Provisional (estimated reading)']"
15,context,monthly_bills,warn,"1 non-Actual bill(s): [['GOA', '2026-07']]"
2,meter,goa_hourly,warn,24 fully duplicated rows
3,meter,goa_hourly,warn,24 duplicate keys on ['timestamp']
4,meter,goa_hourly,warn,24 duplicate timestamps
0,meter,gurugram_15min,warn,145 null cells in 1 column(s): ['power_factor']
5,meter,gurugram_15min,warn,"3 meter-fault run(s) (>=1 h) in grid_kwh/kvah/kVA (['12 Jan', '08 Jul', '17 Aug']) -> mask from baseline; kw_avg and PF stay live and track the sub-meters (checked), so the billed columns are corrupted, not the load; rebuild them from kw_avg"
1,meter,jaipur_hourly,warn,"50 null cells in 5 column(s): ['grid_kwh', 'grid_kvah', 'kw_avg', 'kva_max_in_hour', 'power_factor']"


{'warn': 11, 'info': 6}
